# GPU Evaluation — SAMURAI streaming trên NVIDIA Tesla T4

Notebook đánh giá hiệu năng **headless** (ADR-0002): chạy pipeline trên GPU, xuất
- `data/out_gpu.mp4` — video đã overlay (mask + ô bao + quỹ đạo + HUD)
- `data/gpu_metrics.csv` / `data/gpu_summary.json` — số liệu từng frame + tổng hợp

> Số trong `gpu_summary.json` là **số đo GPU thật**, dùng cho mục "Đánh giá hiệu năng" của báo cáo,
> đặt cạnh số CPU laptop để đối chiếu. Không dùng số GPU để nói "demo live realtime".

**Cách dùng nhanh nhất:** Runtime → Change runtime type → **T4 GPU** → Save → chạy cell **⚡ CHẠY TẤT CẢ**.
Muốn chạy từng bước để hiểu chi tiết thì chạy lần lượt các cell bên dưới.


In [3]:
# 1) Kiểm tra GPU — phải thấy Tesla T4
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA sẵn sàng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("Chưa bật GPU: Runtime → Change runtime type → T4 GPU")


Thu Oct 22 15:55:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# ⚡ CHẠY TẤT CẢ (1 cell): clone/pull repo → cài đặt → tải checkpoint → tạo video →
# chạy pipeline GPU (image_size 512 + reanchor) → in số liệu → đóng gói kết quả.
# Chỉ cần bật GPU trước: Runtime → Change runtime type → T4 GPU → Save
import os, subprocess
os.chdir("/content")
if not os.path.isdir("DoAn-NAB"):
    subprocess.run(["git", "clone", "-q", "https://github.com/BinhNguyen1912/DoAn-NAB.git"], check=True)
os.chdir("DoAn-NAB")
subprocess.run(["git", "pull", "-q"], check=False)
!python scripts/colab_run_all.py


[setup] package sam2 đã cài
[setup] checkpoint OK: 156.0 MB
[setup] sinh video test 1280x720 (60 frame)…
[setup] video test: bbox frame 0 = (200, 200, 320, 320)
[run] /usr/bin/python3 tools/run_live.py --source /content/DoAn-NAB/data/input.mp4 --device cuda --image-size 512 --record data/out_gpu.mp4 --metrics-csv data/gpu_metrics.csv --metrics-summary data/gpu_summary.json --reanchor-every 10 --bbox 200,200,120,120
[setup] $ /usr/bin/python3 tools/run_live.py --source /content/DoAn-NAB/data/input.mp4 --device cuda --image-size 512 --record data/out_gpu.mp4 --metrics-csv data/gpu_metrics.csv --metrics-summary data/gpu_summary.json --reanchor-every 10 --bbox 200,200,120,120
/content/DoAn-NAB/samurai/sam2/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
[pipeline] model image_size=512 (nhanh hơn trên CPU, đánh đổi chi tiết vật thể nhỏ)
[pipel

In [5]:
# 2) Lấy code + SAMURAI + checkpoint (khoảng 2-4 phút)
!git clone -q https://github.com/BinhNguyen1912/DoAn-NAB.git
%cd DoAn-NAB
!git clone -q --depth 1 https://github.com/yangchris11/samurai.git samurai
!mkdir -p samurai/sam2/checkpoints
# Tải checkpoint bằng PYTHON thuần (không dùng !wget) để chạy được ở bất kỳ cell Python nào
import os
import urllib.request

_ckpt = "samurai/sam2/checkpoints/sam2.1_hiera_tiny.pt"
_url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt"
if (not os.path.exists(_ckpt)) or os.path.getsize(_ckpt) < 100_000_000:
    print("đang tải checkpoint (~156 MB)…")
    urllib.request.urlretrieve(_url, _ckpt)
!ls -lh samurai/sam2/checkpoints/

# Colab ĐÃ có numpy/opencv/pillow/torch-CUDA. KHÔNG cài lại numpy/opencv để tránh
# làm hỏng môi trường Colab (pip sẽ đòi restart runtime). Chỉ cài phần còn thiếu:
# loguru: SAMURAI import trong sam2/modeling/sam2_base.py (SAM 2 gốc không cần)
!pip install -q pyyaml loguru scipy   # scipy: sam2/utils/kalman_filter.py (SAMURAI)
!pip install -q -e samurai/sam2
import cv2
import numpy as np

print("opencv", cv2.__version__, "| numpy", np.__version__)
print("cài đặt xong")
_size = os.path.getsize(_ckpt)
assert _size > 100_000_000, f"checkpoint tải HỎNG: {_size} bytes — kiểm tra mạng rồi chạy lại cell này"
print(f"checkpoint OK: {_size/1e6:.1f} MB")


/content/DoAn-NAB/DoAn-NAB
đang tải checkpoint (~156 MB)…
total 149M
-rwxr-xr-x 1 root root 2.6K Oct 22 16:00 download_ckpts.sh
-rw-r--r-- 1 root root 149M Oct 22 16:00 sam2.1_hiera_tiny.pt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
opencv 4.14.0 | numpy 2.1.3
cài đặt xong
checkpoint OK: 156.0 MB


## 3) Chuẩn bị video đầu vào

**Cách A (khuyến nghị):** quay video bằng điện thoại (10–15 giây, người/vật chuyển động),
rồi kéo file vào panel **Files** bên trái (thư mục `/content/DoAn-NAB/data/`), đổi tên `input.mp4`.

**Cách B:** chạy cell dưới để sinh video test tổng hợp (ô vuông di chuyển) — dùng để kiểm tra
notebook chạy được trước khi có video thật.


In [6]:
# 3b) Sinh video test nếu chưa có input.mp4 (ô vuông cam di chuyển, 60 frame)
import os
if not os.path.exists("data/input.mp4"):
    from src.synthetic_video import generate_tracking_sequence
    import cv2
    fx = generate_tracking_sequence(num_frames=60, frame_size=(720, 1280), square_size=120,
                                    start=(200, 200), velocity=(14, 6))
    out = cv2.VideoWriter("data/input.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 25, (1280, 720))
    for f in fx.frames:
        out.write(f)
    out.release()
    print("đã sinh data/input.mp4 (ô vuông tại frame 0:", fx.bboxes[0], ")")
else:
    print("đã có data/input.mp4")
import cv2
cap = cv2.VideoCapture("data/input.mp4")
print("video:", int(cap.get(3)), "x", int(cap.get(4)), "|", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), "frame")
cap.release()


đã sinh data/input.mp4 (ô vuông tại frame 0: (200, 200, 320, 320) )
video: 1280 x 720 | 60 frame


## 4) Chạy pipeline trên GPU

`--bbox` là ô chọn đối tượng ở **frame 0** (x,y,w,h). Với video tự sinh ở trên dùng `200,200,120,120`;
với video thật, xem frame 0 để chọn toạ độ (hoặc bỏ `--bbox` để dùng ô giữa khung).


In [10]:
!python tools/run_live.py --source data/input.mp4 --bbox "200,200,120,120" --device cuda --record data/out_gpu.mp4 --metrics-csv data/gpu_metrics.csv --metrics-summary data/gpu_summary.json


/content/DoAn-NAB/DoAn-NAB/samurai/sam2/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
[pipeline] device=cuda (Tesla T4)
SAMURAI mode: True
[pipeline] ghi video: data/out_gpu.mp4 (25 fps)
[pipeline] bắt đầu tracking... (headless)
[pipeline] ghi metrics: data/gpu_metrics.csv (+summary JSON nếu --metrics-summary)
frame loading (JPEG): 100% 1/1 [00:00<00:00, 22.07it/s]
[metrics] 60 frame | mean=343 ms | p95=352 ms | mất dấu 0/60 (0%) | score tb=7.33
[kết quả] 60 frame | mean=343 ms | p95=352 ms | FPS hiệu dụng ≈ 2.92
(GPU Tesla T4 — số liệu thực đo theo ADR-0002)


In [8]:
# 5) Xem số liệu + video kết quả ngay trong notebook
import json
from IPython.display import Video, display
s = json.load(open("data/gpu_summary.json"))
print("=== SỐ LIỆU GPU (thật) ===")
for k, v in s.items():
    print(f"  {k:20} {round(v,3) if isinstance(v, float) else v}")
display(Video("data/out_gpu.mp4", embed=True, width=720))


=== SỐ LIỆU GPU (thật) ===
  total_frames         60
  mean_latency_ms      380.899
  p95_latency_ms       348.029
  effective_fps        2.625
  mean_score           7.328
  lost_count           0
  lost_ratio           0.0
  reacquiring_frames   0
  reacquired_count     0


In [9]:
# 6) Tải TẤT CẢ file kết quả (mọi lần chạy: gpu_*, *_2, out_gpu*)
import glob
from google.colab import files

paths = sorted(glob.glob("data/out_gpu*.mp4") + glob.glob("data/gpu_*.csv") + glob.glob("data/gpu_*.json"))
if not paths:
    print("Chưa có file kết quả — chạy cell pipeline trước.")
for pth in paths:
    print("tải:", pth)
    files.download(pth)


tải: data/gpu_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

tải: data/gpu_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

tải: data/out_gpu.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7) Ghi chú cho báo cáo

- Đặt **số GPU** (cell 5) cạnh **số CPU laptop** (`data/webcam_summary.json` / bảng trong README
  mục "Đánh giá hiệu năng") — nêu rõ phần cứng từng bên.
- Câu trả lời cho hội đồng: *hệ thống real-time-capable trên GPU (số đo T4); trên laptop CPU
  chỉ đạt ~0.2–2 FPS nên demo live là slideshow có chủ đích (ADR-0002).*
- Nếu muốn video mượt hơn cho slide: quay video nguồn 10–15 giây, chạy lại cell 4 — GPU xử lý
  ~15–25 FPS, thời gian chạy chỉ khoảng 20–30 giây.
